In [ ]:
# Adding LLM as an agent for understanding the user query and calling the tool
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "openai/gpt-oss-120b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,   # explicit dtype (faster + smaller than "auto"/float32)
    device_map="auto",
    attn_implementation="sdpa",  # faster attention kernel, no extra install needed
)

config.json:   0%|          | 0.00/2.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/tqdm.py", line 440, in hf_thread_map
    for future in as_completed(future_to_index):
                  ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/concurrent/futures/_base.py", line 243, in as_completed
    waiter.event.wait(wait_timeout)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/threading.py", line 660, in wait
    signaled = self._cond.wait(timeout)
  File "/usr/lib/python3.13/threading.py", line 359, in wait
    waiter.acquire()
    ~~~~~~~~~~~~~~^^
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_511/3609482234.py", line 6, in <cell line

TypeError: object of type 'NoneType' has no len()

In [ ]:
from langchain_core.tools import tool
import json


In [ ]:
# 1. weather tool - now using a real-time weather API (Open-Meteo, free, no API key needed)
import requests

@tool
def weather(city: str):
  """Returns the current real-time weather for a given city."""
  try:
    # step 1: turn the city name into latitude/longitude
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo_res = requests.get(geo_url, params={"name": city, "count": 1}).json()
    if not geo_res.get("results"):
      return "weather data not found"

    lat = geo_res["results"][0]["latitude"]
    lon = geo_res["results"][0]["longitude"]

    # step 2: fetch the current weather for that location
    weather_url = "https://api.open-meteo.com/v1/forecast"
    weather_res = requests.get(weather_url, params={
        "latitude": lat,
        "longitude": lon,
        "current_weather": True
    }).json()

    current = weather_res.get("current_weather", {})
    temp = current.get("temperature")
    wind = current.get("windspeed")
    return f"{temp}C, wind {wind} km/h"
  except Exception as e:
    return f"weather data not found: {e}"


In [ ]:
# City info tool - now pulling a real summary from Wikipedia instead of a hardcoded dict
@tool
def Cityinfo(city: str):
  """Returns real city information for a given city, pulled from Wikipedia."""
  try:
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + city.strip().replace(" ", "_")
    headers = {"User-Agent": "MyTravelAgent/1.0 (myemail@example.com)"}
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
      return "city info data not found"

    data = res.json()
    return data.get("extract", "city info data not found")
  except Exception as e:
    return f"city info data not found: {e}"


In [ ]:
# Currency exchange tool - now using a real-time free API (frankfurter.app, backed by the European Central Bank)
@tool
def currency(amount: float, from_currency: str, to_currency: str):
  """Convert the amount from one currency to another using live exchange rates."""
  try:
    url = "https://api.frankfurter.app/latest"
    res = requests.get(url, params={"from": from_currency, "to": to_currency})
    if res.status_code != 200:
      return "Conversion not available"

    data = res.json()
    rate = data.get("rates", {}).get(to_currency)
    if rate is None:
      return "Conversion not available"

    return amount * rate
  except Exception as e:
    return f"Conversion not available: {e}"


In [ ]:
# store all the tools in a list (useful if you later want to bind
# them straight to an LLM, e.g. model.bind_tools(tool_list))


In [ ]:
# cross check an example
print(weather.invoke({"city":"Udaipur"}))
print(Cityinfo.invoke({"city":"Udaipur"}))
print(currency.invoke({"amount":100, "from_currency":"USD", "to_currency":"INR"}))

In [ ]:
# tools store to dictionary
tools={"weather": weather, "Cityinfo": Cityinfo, "currency": currency}#

In [ ]:
! pip install -q langchain langchain-huggingface

In [ ]:
! pip install -q langchain-core langchain-groq

In [ ]:
# before built with lang graph create langchain chat pipeline
# from langchain_huggingface import HuggingFacePipeline,ChatHuggingFace
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = "api key "
chat_model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0)

In [ ]:
tools_list = [weather, Cityinfo, currency]
llm_with_tools =chat_model.bind_tools(tools_list)

In [ ]:
# now lang graph architecture
from langgraph.graph import StateGraph, MessagesState, START,END
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
def agent_node(state:MessagesState):
  response = llm_with_tools.invoke(state["messages"])
  return {"messages":[response]}

In [ ]:
tool_node = ToolNode(tools_list)

In [ ]:
graph = StateGraph(MessagesState)
graph.add_node("agent",agent_node)
graph.add_node("tools",tool_node)
graph.add_edge(START,"agent")
graph.add_conditional_edges("agent",tools_condition)
graph.add_edge("tools","agent")
graph.add_edge("agent",END)
travel_agent = graph.compile()


In [ ]:
# display the graph arch
from IPython.display import Image, display
display(Image(travel_agent.get_graph().draw_mermaid_png()))

In [ ]:
# invoke responses
from langchain_core.messages import HumanMessage, SystemMessage
system_msg = SystemMessage(content="""you area travel agent. use available tools
which are required to answer the user's question""")
def run_travel_agent(prompt):
  result = travel_agent.invoke({"messages":[system_msg,HumanMessage(content=prompt)]})
  return result["messages"][-1].content


In [ ]:
run_travel_agent("how is weather in Bangalore")

In [ ]:
result = travel_agent.invoke({"messages":[system_msg,HumanMessage(content="how is weather in Bangalore")]})
for msg in result["messages"]:
  print(type(msg), msg.content)